In [1]:
"""
================================================================================
PROYECTO: IA-Energy-Alternatives
MÓDULO 02: Simulación de Red Espín-Fonón para Aleaciones Magnetocalóricas (Vector B)
GRUPO: Dark Shadows Group
================================================================================

JUSTIFICACIÓN TÉCNICA Y FÍSICA (¿POR QUÉ HACEMOS ESTO?):

1. EL PRINCIPIO MAGNETOCALÓRICO:
   Para eliminar el enfriamiento por evaporación de agua, modelamos materiales 
   sólidos que cambian su entropía (y temperatura) bajo un campo magnético externo. 
   Para diseñar estas aleaciones (como Ni-Ti), necesitamos predecir cómo responde 
   su red de espines electrónicos antes de fabricar el material en un laboratorio.

2. LA SOLUCIÓN CUÁNTICA (MODELO DE ISING TRANSVERSAL):
   Modelamos una cadena de espines acoplados magnéticamente. El término J representa 
   la energía de intercambio entre átomos vecinos (estructura cristalina), y el 
   término g_field representa el campo magnético externo aplicado. Calculamos el 
   estado de mínima energía (Ground State) variando el campo magnético para medir 
   la brecha energética que induce el cambio térmico.

3. LIMITACIÓN Y ESCALA:
   Usamos un modelo de 4 sitios (sitios atómicos). En hardware clásico, mapear sistemas 
   grandes sufre por el crecimiento exponencial de la matriz del Hamiltoniano (2^N). 
   Este bloque de Qiskit deja la estructura lista para escalar a N-sitios en simuladores.
================================================================================
"""

import sys
sys.path.append('..')
from config.quantum_settings import get_local_quantum_backend

from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import NumPyMinimumEigensolver

# 1. PARAMETRIZACIÓN FISICA DE LA RED METÁLICA
num_sites = 4      # Número de nodos cuánticos en la red cristalina
J = 1.0            # Constante de acoplamiento ferromagnético
g_field = 0.5      # Magnitud del campo magnético transversal perturbador (B)

pauli_list = []
coeffs = []

# 2. CONSTRUCCIÓN DEL HAMILTONIANO DE ISING (Interacción Z_i X Z_{i+1})
# Representa la energía interna debido al alineamiento de espines vecinos
for i in range(num_sites - 1):
    op_str = ["I"] * num_sites
    op_str[i] = "Z"
    op_str[i+1] = "Z"
    pauli_list.append("".join(op_str))
    coeffs.append(-J)

# 3. COMPONENTE TRANSVERSAL (Campo magnético externo X)
# Representa la fuerza externa que intenta desalinear los espines de la red
for i in range(num_sites):
    op_str = ["I"] * num_sites
    op_str[i] = "X"
    pauli_list.append("".join(op_str))
    coeffs.append(-g_field)

# Ensamble final del operador tensorial de Pauli
lattice_hamiltonian = SparsePauliOp(pauli_list, coeffs)

# 4. RESOLUCIÓN DIAGONALIZADA (Benchmarking exacto)
exact_solver = NumPyMinimumEigensolver()
result = exact_solver.compute_minimum_eigenvalue(lattice_hamiltonian)

print(f"[OK] Simulación de red metálica magnetocalórica completada.")
print(f"-> Campo Magnético Aplicado (g): {g_field} Tesla equivalente.")
print(f"-> Energía Libre del Estado Fundamental: {result.eigenvalue.real:.4f} unidades energéticas.")

[OK] Simulación de red metálica magnetocalórica completada.
-> Campo Magnético Aplicado (g): 0.5 Tesla equivalente.
-> Energía Libre del Estado Fundamental: -3.4270 unidades energéticas.
